# Lab 9: Time Series Analyses (Lectures 17 - 18)

<table>
<tr>
<td><img src="images/aria_poker.webp" width="355"></td>
<td><img src="images/venetian_slots.png" width="550"></td>
<td><img src="images/craps_table.jpg" width="400"></td>
</tr>
</table>

In [ ]:
# Please run this initialization cell
library(ottr)
library(readr)
library(knitr)
library(dplyr)
library(rlang)
library(ggplot2)
library(forecast)
library(lmtest)
options(repr.plot.width=20, repr.plot.height=8)
theme_update(text = element_text(size=20))

## Part 1: Getting Started
First, review the [Lucas 2013](https://dsc152.com/resources/references/Lucas2013_poker-slots-table.pdf) paper that was introduced in lecture. Then answer the following introductory/review questions.

**Question 1.1.** Which of the following best describes the overall goal of the Lucas 2013 paper?

1. To determine the extent to which slot machines and table games make money for a casino
2. To determine the extent to which poker rooms alone make money for a casino
3. To determine whether poker rooms generate more profit than slot machines and table games for a casino
4. To determine the extent to which poker rooms drive traffic to slot machines and table games at a casino
5. To determine how much more profitable slot machines and table games are for a casino than a poker room

Assign the variable `goal` to the value of 1, 2, 3, 4 or 5.

In [ ]:
goal <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_1.R")

**Question 1.2.** Which of the following best describes the overall shortcoming of the Lucas 2013 paper?

1. The data fails the independence condition for linear regression.
2. The author's workflow inappropriately mixes model selection with statistical inference.
3. The author used a time series analysis when standard linear regression would have sufficed.
4. The author's scientific question is ill-posed and does not translate to a valid statistical question.

Assign the variable `shortcoming` to the value of 1, 2, 3 or 4.

In [ ]:
shortcoming <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** Consider this statement on pg. 53 of the paper: "To ease interpretation of the final model results, all continuous independent variable values were also converted to their natural log form, creating double-log models. A technique common in econometric modeling (Dielman, 1996), the double-log model features regression coefficients which are expressed as elasticities, i.e., the elasticity of Y with respect to X (Kahane, 2008, p. 84)."

Which of the following is most accurate with regard to this statement?

1. This is a reasonable justification for performing a double-log transformation.
2. The primary reason for variable transformation in linear regression should not be for ease of interpretation, but rather should be to improve the linear fit between the outcome variable and the covariate(s).

Assign the variable `d_log` to 1 or 2. 

In [ ]:
d_log <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_3.R")

## Part 2: Loading in the data
Now, let us load in the data. Recall that, as the actual data are proprietary, the data we will use were generated by Claude under directions to make output from them match the graphs and tables in the Lucas 2013 paper as closely as possible. Note that there are still some discrepancies that could not be resolved.

Run the code cell below to load in the data and display the first several rows.

In [ ]:
# Run this code cell
casino <- read_csv("casino_poker_data.csv", show_col_types=FALSE)
casino[1:8,]

**Question 2.1.** Take a look at the dataframe and the descriptions of the variables in the paper, and then match each of the following column names of the dataframe to its correct description among the choices below them:

`dow`, `NCAABBALL`, `R1_COIN`, `R2_RAKE`, `R3_DROP`

1. The amount of money earned by the casino on slot machines in Resort 1 on any given day.
2. The amount of money earned by the casino on slot machines in Resort 2 on any given day.
3. The amount of money earned by the casino on slot machines in Resort 3 on any given day.
4. The total amount of money wagered by customers on slot machines in Resort 1 on any given day.
5. The total amount of money wagered by customers on slot machines in Resort 2 on any given day.
6. The total amount of money wagered by customers on slot machines in Resort 3 on any given day.
7. A mysterious measure of business volume on table games in Resort 1 on any given day.
8. A mysterious measure of business volume on table games in Resort 2 on any given day.
9. A mysterious measure of business volume on table games in Resort 3 on any given day.
10. The amount of money earned by the poker room in Resort 1 on any given day.
11. The amount of money earned by the poker room in Resort 2 on any given day.
12. The amount of money earned by the poker room in Resort 3 on any given day.
13. An indicator variable for whether NCAA College Basketball Championship games were occurring on that day.
14. An indicator variable for whether the Kentucky Derby was occurring that day.
15. The day of the week

Store your answer as a vector into the variable `col_desc`; for example, the answer `c(9,1,3,4,2)` would indicate that:
- `dow` is the 9th choice above
- `NCAABBALL` is the 1st choice above

and so on.

In [ ]:
var_desc <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_1.R")

**Question 2.2.** According to the scientific question as described in the paper, which of the variable(s) from the above list is/are our primary predictor variable(s) of interest? Store your response in the variable `primary` below. If there is more than one, store your answer as a vector in ascending order.

In [ ]:
primary <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_2.R")

### What resorts are these actually?
The paper identifies the three casino properties in this study simply as "Resort 1," "Resort 2" and "Resort 3." But in case you were curious like I was, I did some digging to see which resorts they are most likely to be and wanted to share what I found. The following conclusions are based on a combination of my memory, Google searches, and Claude.

Note: if you are not interested in any of this, you may skip down to **Part 3** below, as there are no questions to answer here.

---

**Resort 1:** According to the paper, this resort was on the Las Vegas strip (the main road in Las Vegas that consists of many of the most popular casinos), had over 3,500 hotel rooms at mid to low-end price points, and 8 poker tables at the time of data collection (which as a reminder, was Feb to Sept of 2009).
 - My initial guess was **Bally's Hotel & Casino** (which has since been rebranded as the new location of the Horseshoe Las Vegas). Admittedly, this was a biased guess as this was my go-to property for lodging during my semi-broke grad school years so it came to mind primarily due to my familiarity with it. The main issue with this guess is that it does not appear to have had over 3,500 hotel rooms (it appears to have maxed out at approximately 2,800).
 - Claude's first guess was **Excalibur Hotel & Casino.** The only problem with this one is that in 2008 into mid-2009, they were experimenting with electronic dealer-less poker tables and it is well documented that they had 12 of them (not 8), until they abandonded the electronic tables in July 2009 at which point it is unclear how many poker tables they had, though 8 is plausible (but they still would have had 12 during the data collection period until that point, which is not what was described).
 - My final guess is **Luxor Hotel & Casino** (pictured below). It has 4,407 total hotel rooms and had 9 poker tables when the poker room was operational, according to most historical webpages and also [this one](https://vegasadvantage.com/open-las-vegas-poker-rooms/closed/). But 9 is not that far from 8, so it is plausible that it had 8 during the year of data collection.

<table>
<tr>
<td><img src="images/luxor.jpg" width="525"></td>
<td><img src="images/luxor_poker.png" width="450"></td>
</tr>
</table>

---

**Resort 2:** The paper states that this one was also on the strip with over 3,500 hotel rooms, but at a mid to high-end price range, and a poker room with 22 poker tables at the time of data collection.
 - A natural guess in some ways is **The Bellagio,** but its poker room has always been too big to fit the description of 22 tables. Claude also thinks it is too high-end; I think that's debatable, but the size of the poker room is definitely a problem with the fit.
 - My first thought actually was **The Mirage,** which fits reasonably well except that web sources state that its poker room had 19 tables (close to 22, but it's still a discrepancy). 
 - Claude is extremely confident (even with repeated questioning and probing from me) that the answer is the **MGM Grand** because it fits the hotel room count, satisfies "mid to high-end," and a few sources state that it has exactly 22 poker tables. Part of me doesn't want to let go of **The Mirage** (in no small part due to the prevailing sentiment in the mid-to-late 1990s, summed up best by Matt Damon's quote in Rounders, "*The poker room at The Mirage in Vegas is the center of the poker universe*") but I think **MGM Grand** (pictured below) is certainly a reasonable guess and probably is the most likely, all things considered.

<table>
<tr>
<td><img src="images/MGM.png" width="500"></td>
<td><img src="images/mgm_poker.jpg" width="450"></td>
</tr>
</table>

---

**Resort 3:** This one is off-strip, had less than 750 hotel rooms and is a locals joint according to the paper. It had fewer restaurants and entertainment options, and its poker room had 12 tables.
 - My mind went to **The Orleans,** **Boulder Station** and **Red Rock Casino.** However, upon inspection, all of these have large issues of fit for a variety of reasons.
 - The best bet is probably **Sam's Town** (pictured below). It fits the hotel room count and is solidly known as an off-strip locals joint, with the described level of amenities. Claude is expressing somewhat strong uncertainty about it to me because it (and I) can only find web history stating that it has exactly 11 (not 12) poker tables in their poker room, but I cannot find anything else to fit any better than this.

<table>
<tr>
<td><img src="images/Sams.jpg" width="600"></td>
<td><img src="images/sams_poker.jpg" width="450"></td>
</tr>
</table>

---

## Part 3: A Proper Statistical Inference Workflow
Now, let us walk through a complete and proper statistical inference workflow on these data from the three casinos. Many aspects of this workflow that we will do will differ from what was done in the Lucas 2013 paper; some of the differences are debatable, whereas in some other differences we will highlight what actually should have been done instead of what was done in Lucas 2013. 

As emphasized in Lecture \#13, if we are doing statistical inference then the first thing that we must do is to decide on the pre-specified statistical model for that inference. First, I will choose to have us keep the variables on their raw scale instead of taking natural logs. Next, I will take the standpoint that the time series nature of the data can be entirely captured by the day of the week; that is, based on the observation that certain days (such as weekend days) will tend to have more business in general, this makes "day of the week" a confounding variable. Setting Monday as the baseline day, the following would be our model for inference, e.g. for `DROP`:

$$
\widehat{DROP} = \hat{\beta}_0 + \hat{\beta}_1 \cdot RAKE + \hat{\beta}_2 \cdot TUES + \hat{\beta}_3 \cdot WED + \hat{\beta}_4 \cdot THURS + \hat{\beta}_5 \cdot FRI + \hat{\beta}_6 \cdot SAT + \hat{\beta}_7 \cdot SUN
$$

**Question 3.1.** Which of the following do you think is/are correct regarding the difference in this model vs. any of the "final models" in Lucas 2013?

1. It is objectively correct to omit the AR and MA terms from the regression model.
2. The decision regarding whether to log-transform any variables for the inference model should be determined by examining the dataset that you are using for inference, and checking whether transformations would improve the linear fit of the model.
3. The final models used for inference in the Lucas 2013 paper were obtained by performing variable selection, and this is objectively incorrect to do.
4. Day of the week is the only possible confounding variable, so our choice of model for inference above is definitively the correct one.
5. From a scientific rationale standpoint (i.e. using scientific rationale for determining the pre-specified statistical model for inference), it would be difficult to come up with justification for why each of the six models run in Tables 3 and 4 of Lucas 2013 should be different from each other.

Store your choice into the variable `inf_model` below. If multiple choices are correct, store your answer as a vector of values in ascending order.

In [ ]:
inf_model <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_1.R")

### Tabular and Graphical Summaries of the data

**Question 3.2.1.** Now, the next thing to do is to make summary tables and graphical summaries of the data. First, consider Table 1 on pg. 55 in the Lucas 2013 paper. All of the values in this table are (natural) log transformed. Let us make a version of the table that is on the original scale of the variables (actual dollars), to be consistent with our model for inference. For now, ignore the `TREND` variable for Resort 3. It is also acceptable to keep summary statistics unadjusted for confounders; that is, we can present overall summary statistics as was done in Lucas 2013, as opposed to presenting them in subgroups by day (the table would get very large if we did this, and the added information would not be particularly illuminating with regard to whether there is actually a relationship between `RAKE` and either `COIN-IN` or `DROP`). 

To summarize: write R code to create a table similar to that of Table 1 on pg. 55 of the Lucas 2013 paper, but instead of all values on the natural log scale, express them on their original scale. 

There are many ways to do this, and your code may take up as many lines as you need; looping or using something like `apply` or `sapply` may shorten your code, but you are also welcome to brute force it. Either way, your final answer should be stored in the variable `summary_usd` as a $9 \times 4$ matrix with the same row and column headings as the paper's Table 1. Again, ignore the `TREND` variable for Resort 3 for now.

In [ ]:
summary_usd <- matrix(NA, nrow=9, ncol=4)
rownames(summary_usd) <- c("R1 COIN-IN", "R1 DROP", "R1 RAKE", "R2 COIN-IN", "R2 DROP", "R2 RAKE", "R3 COIN-IN", "R3 DROP", "R3 RAKE")
colnames(summary_usd) <- c("Mean", "Std. Dev", "Min.", "Max.")

# YOUR CODE HERE

summary_usd

In [ ]:
. = ottr::check("tests/q3_2_1.R")

Notice a couple of things about the table:

 - COIN-IN values are the largest of the three metrics across all resorts. However, while it is true that slot machines are generally one of a casino's biggest money makers, the values of COIN-IN here are somewhat misleading as these are the total amounts of money that customers put into machines, and: 1) They won't always lose all of it; 2) Sometimes they will even win (meaning that the casino loses). On average, a casino can expect to keep appproximately 7.5\% of these values (as noted on pg. 48 of Lucas 2013).
 - DROP is a bit nebulous, as stated above. It seems to be neither the total amount wagered, nor the total amount that the casino earned from table games. Nevertheless, according to this [UNLV source](https://gaming.library.unlv.edu/reports/nv_table_hold.pdf), it appears that approximately 15\% of the `DROP` values are kept as profit, on average.
 - The RAKE values are in fact the amount that the casino poker room earned from the poker tables on any given day.

**Question 3.2.2.** With this in mind, let us make new versions of `COIN-IN` and `DROP` that represent the estimated profit from slot machines and table games, respectively. Add them as new columns in the `casino` dataframe named `R1_COIN_pf`, `R1_DROP_pf`, etc., and then make a new, final **Table 1** with these values, stored in the new variable `summary_usd_profit`, again as a $9 \times 4$ matrix.

Note that this step will not impact any of the inference at all, as we are effectively just scaling each of these variables. But after doing this step, all of these variables will now represent actual profit in dollars that the casino makes each day from each class of games.

*Hint: to make the new table, you can either use the new variables that you created, or you can directly scale the values in the original `summary_usd` table, as either will work here.*

In [ ]:
# First make new versions of each variable as new columns in dataframe
# YOUR CODE HERE


# Then make new summary table
summary_usd_profit <- matrix(NA, nrow=9, ncol=4)
rownames(summary_usd_profit) <- c("R1 COIN-IN", "R1 DROP", "R1 RAKE", "R2 COIN-IN", "R2 DROP", "R2 RAKE", "R3 COIN-IN", "R3 DROP", "R3 RAKE")
colnames(summary_usd_profit) <- c("Mean", "Std. Dev", "Min.", "Max.")

# YOUR CODE HERE

summary_usd_profit

In [ ]:
. = ottr::check("tests/q3_2_2.R")

From this new Table 1, we can more directly compare revenue from each type of game, and we observe that slot machine revenue is still generally the highest, followed by table games, and finally the poker room.

Note that one thing we are still not accounting for here is operating expenses, particularly as table games and the poker room require human staffing, whereas slot machines are much more self-sufficient. We will not attempt to account for that here, as this is difficult to quantify accurately.

**Question 3.3.** Now let's make a graphical summary. The study presented a total of 6 statistical tests (3 resorts, 2 outcome variables for each), but for this exercise we will focus on the relationship between `RAKE` and `COIN_pf` in Resort 1 aka maybe **Luxor Hotel & Casino** (chosen somewhat arbitrarily; I just didn't want us to have to go through all 6 as that would be a bit cumbersome). In Table 3 on pg. 57 of Lucas 2013, this relationship on the double-log scale showed a p-value of 0.0959 in the final model, so $H_0$ was not rejected. 

Let us make a scatterplot of the relationship between rake and slot machine revenue in Resort 1, on the original scale of dollars of profit for the casino. For this question, do not worry about adding axes labels, a title, or re-sizing anything. Also, like in the summary tables above, do not worry about adjusting for day of the week (or any other covariate).

Then, after making the graph, select all the following that best matches on your observation:
1. There does not appear to be any relationship between `RAKE` and `COIN_pf` in Resort 1.
2. There appears to be a negative linear relationship between `RAKE` and `COIN_pf` in Resort 1.
3. On average, it appears that a `$2500` increase in poker room profit is associated with approximately a `$25,000` increase in slot machine profit, before adjusting for day of the week.
4. There may be some slight concern about heteroskedasticity.
5. There appear to be large concerns about non-linearity in the relationship between `RAKE` and `COIN_pf`.

Store your response into the variable `graph1_obs` below, after making the graph. If multiple are correct, store them in a vector in ascending order. 


In [ ]:
# First, make the graph
# YOUR CODE HERE

In [ ]:
# Then, store your choice(s) from the question above.
graph1_obs <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_3.R")

### Primary Statistical Analysis

**Question 3.4.1.** Now let's run the primary analysis. First, we'll set `Monday` as the baseline level (code provided); otherwise, R would choose Friday alphabetically. While this doesn't matter that much, it still makes a bit more sense for it to be Monday. Then, run the primary model from above, and store the relevant coefficient estimate and p-value into `coef` and `pval` in the code below.

In [ ]:
casino$dow <- relevel(as.factor(casino$dow), ref="Monday")

model1 <- NULL # YOUR CODE HERE

# display the model output
summary(model1)

coef <- NULL # YOUR CODE HERE
pval <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_4_1.R")

**Question 3.4.2.** Which of the following are true based on the output that you observed from the primary analysis?

1. The estimated profit from slot machines in Resort 1 on a Monday with no poker revenue is `$132669.654`.
2. Every increase in poker room traffic that results in a `$1` increase in poker room profit for the casino is associated with an expected increase of `$2.449` in slot machine profit for the casino.
3. Since `dowThursday` and `dowTuesday` do not have p-values less than 0.05, they should be removed from the model and the model should be re-run without them for inference.
4. This output suggests that Saturdays are the busiest day for slot machines at Resort 1.

Store your choice into the variable `prim_obs` below; if multiple options are correct, store them as a vector of values in ascending order.


In [ ]:
prim_obs <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_4_2.R")

### Model Diagnostics for Checking Conditions for Validity

Here we will perform the model diagnostics as outlined in Lecture \#10.

**Question 3.5.1** First, let's check the linearity condition. Make an appropriate residual plot below, and then indicate whether the linearity condition appears to be violated or not based on it.

In [ ]:
# Make the appropriate residual plot here
# YOUR CODE HERE

In [ ]:
# Then indicate here: 0=likely violated, 1=maybe violated, 2=likely not violated
lin_cond <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_5_1.R")

**Question 3.5.2** Next, let's check the independence condition, by checking for a temporal trend. Make an appropriate residual plot below, and then indicate whether the independence condition, as a time trend, appears to be violated or not based on it.

In [ ]:
# Make the appropriate residual plot here
# YOUR CODE HERE

In [ ]:
# Then indicate here: 0=likely violated, 1=maybe violated, 2=likely not violated
ind_cond <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_5_2.R")

Your residual plot over time should reveal some slight suggestion of cyclical patterns, though it is not nearly as stark as what we will observe on the data below in Part 4, so at least some of the temporal trend has been removed by simply adjusting for day of the week.

**Question 3.5.3.** Thirdly, we'll check the normality condition:

 - Plot a histogram of the residuals
 - Make a QQ-plot
 - Run the Shapiro-Wilk test



In [ ]:
# Make the histogram of residuals
# YOUR CODE HERE

# Make the QQ-plot
# YOUR CODE HERE

# Shapiro-Wilk test
st <- NULL # YOUR CODE HERE
st

In [ ]:
. = ottr::check("tests/q3_5_3.R")

**Question 3.5.4.** Finally, let's check the equal variance condition. Make appropriate residual plots below (recall that it should be with each predictor variable), and then indicate whether the equal variance condition appears to be violated or not based on it.

In [ ]:
# YOUR CODE HERE

In [ ]:
# Then indicate here: 0=likely violated, 1=maybe violated, 2=likely not violated
ev_cond <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_5_4.R")

Now, we also need to perform Sensitivity Analyses as part of the full statistical inference workflow. But, we will move Sensitivity Analyses to Part 4 below, to have an entire section in which we will simultaneously interrogate what was done in the Lucas 2013 paper. 

## Part 4: Sensitivity Analyses along with Comparison of Workflows

As noted in class and above, the primary shortcoming of the Lucas 2013 paper is that it performed the model selection $\rightarrow$ statistical inference workflow that we have discussed as being improper. In this part, we will investigate some facets of that, and other issues.

**Question 4.1.** First, recall from the paper and Part 1 above that all of the analyses in Lucas 2013 were performed with `RAKE`, `COIN-IN` and `DROP` all on the natural log scale. Let us start here by making a table that matches that of Table 1 on pg. 55 of Lucas 2013 nearly identically. Store the result in `summary_log` below. 

*Note: unfortunately we cannot simply take the log of the entire `summary_usd` matrix from **Question 3.2.1** above (Why not? Try it and see what happens), but this question should only require a minor modification to your answer from **Question 3.2.1.***

In [ ]:
summary_log <- matrix(NA, nrow=9, ncol=4)
rownames(summary_log) <- c("R1 COIN-IN", "R1 DROP", "R1 RAKE", "R2 COIN-IN", "R2 DROP", "R2 RAKE", "R3 COIN-IN", "R3 DROP", "R3 RAKE")
colnames(summary_log) <- c("Mean", "Std. Dev", "Min.", "Max.")

# YOUR CODE HERE

summary_log

In [ ]:
. = ottr::check("tests/q4_1.R")

Notice that the values match very very closely to those of Table 1 in Lucas 2013. Good job Claude!

**Question 4.2.** Now let's look at the relationship between `COIN-IN` and `RAKE` on the double-log scale in Resort 1 aka maybe **The Luxor Hotel & Casino.** Make a graph below showing the relationship between `log(RAKE)` and `log(COIN)` in the code cell below. Note that to match the paper, to do this we will use the `R1_COIN` variable, not the `R1_COIN_pf` variable that we created.

Then, select among the following that best describes what you observe (select all that apply).

1. The double-log transformation fixes violations to linearity that were present on the original scale.
2. The double-log transformation fixes concerns about heteroskedasticity that were present on the original scale.
3. From this graph, the slope of a least squares line looks like it might be approximately 0.5, which would indicate that a 1\% increase in `RAKE` is associated with approximately a 0.5\% increase in `COIN-IN`.
4. A reasonable justification for performing a double-log transformation for the primary statistical analysis would be if these data show that model fit is improved over that of the original scale.
5. A reasonable justification for performing a double-log transformation for the primary statistical analysis would be if previous studies had showed that model fit was improved over that of the original scale.

Store your response in the variable `graph_log_obs` below, after making the graph. If multiple options are correct, store them in a vector in ascending order.

In [ ]:
# First, make the graph
# YOUR CODE HERE

In [ ]:
# Then, store your choice(s) from the question above.
graph_log_obs <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q3_3_2.R")

**Question 4.3.** Now, let us explore the time series nature of the data. 

With Resort 1, write R code to make a graph similar to that of the first panel of *Figure 3* on pg. 52 of Lucas 2013, with time on the x-axis, and `log(COIN)` on the y-axis. The x-axis can use `day_num`.

In the code cells below, first make the graph, and then select which of the following best represents what you observe. If multiple are true, store each that are true in ascending order as a vector, into the variable `time_plot` below.

1. The plot shows absolutely no need for modeling these data as time-series data.
2. The plot appears to show a cyclical pattern across time in a similar manner as that of *Figure 3* in the Lucas 2013 paper.
3. The plot shows an increasing trend in `log(COIN)` over time.
4. The plot shows a relationship of any kind between `log(COIN)` and `RAKE`.

In [ ]:
# First, make the plot
# YOUR CODE HERE

In [ ]:
# Then, store your choice(s) from the question above.
time_plot <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q4_3.R")

**Question 4.4.1** Now, in the analysis performed by the author for Resort 1 and `COIN-IN` in Table 3 of Lucas 2013, it shows an "AR(1)" term. 

 - This is short for "auto-regressive with 1 step"
 - If data follow an AR(1) model, this means that the value at each step is a function of the previous step, plus some noise

Let's first look at what data that follow an AR(1) model might look like. Mathematically, a general AR model looks like:

$$
y_t = c + \phi_1 y_{t-1} + \pi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \epsilon_t
$$

where $c$ represents a particular direction of drift, $\phi_i$ are scaling constants, and $\epsilon_t$ is typically $N(0, \sigma^2)$. For an AR(1) model, there would only be $\phi_1$, so it would be:

$$
y_t = c + \phi_1 y_{t-1} + \epsilon_t
$$

Noticing that, in *Figure 3* the value of `log(COIN)` starts around 14.2, let's start a time series at a value of 14.2 with an AR(1) model and see what it looks like. In the code cell below, write code to simulate the following:

 - Start with $y_1 = 14.2$
 - Generate each subsquent $y_i$ according to:
     - $c = 0$
     - $\phi_1 = 1$
     - $\epsilon_t \sim N(0, 1)$
     - Use an accumulator pattern to store these values into a vector `y`
 - Stop when you reach a sample size of 217 (the sample size of the casino data)
 - Plot `y` against `t` where `t` is a vector from 1 to 217.

In [ ]:
set.seed(1) # DO NOT CHANGE THIS!

# YOUR CODE HERE

t <- 1:217
plot(y ~ t, type='l')

In [ ]:
. = ottr::check("tests/q4_4_1.R")

**Question 4.4.2.** Which of the following are true regarding what you observe in the plot that you just made, compared to the first panel of *Figure 3* in Lucas 2013?

1. They look remarkably similar.
2. Whether they look similar or not should be used as justification for whether or not to include an AR(1) term in the primary analysis of a statistical inference workflow.
3. The simulated AR(1) data above drift much further away from the starting point of 14.2 than the `log(COIN)` values for Resort 1 in *Figure 3*.
4. The simulated AR(1) data show a cyclical pattern that could reasonably be explained by day of the week.
5. The `log(COIN)` values for Resort 1 in *Figure 3* show a cyclical pattern that could reasonably be explained by day of the week.

Store your response in the variable `AR1` below. If multiple options are true, store it as a vector of values in ascending order.

In [ ]:
AR1 <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q_4_4_2.R")

**Question 4.5.1.** Now, let us investigate the "final model" for `COIN-IN` in Resort 1, shown in Table 3. In **Question 3.1** above, we already noted that it is hard to justify why the models for inference should be different for each of the three resorts in Table 3 (shown by the fact that different covariates have an "n/a" for each model). 

The reason why Table 3 shows different models for each resort is precisely because the author performed variable selection prior to inference, which, again, we know that we should not do. On that note, it is not described anywhere in the paper exactly how model selection was performed; i.e. was it backward selection with p-values (as we have done in class), forward selection, or some other routine? All that we know is that, in every "final model," the p-value for every term/covariate is less than 0.05, so this seems to indicate that the variable selection was at least in some manner guided by p-values.

Regarding this final model for `COIN-IN` at Resort 1 in Table 3, which of the following are true?

1. Although the rationale for choosing to log-transform both `COIN-IN` and `RAKE` is unsatisfactory (see **Question 1.3** above), at least it appears to be determined *a priori* (prior to running the primary analysis), which is the correct thing to do.
2. The "Constant" is the intercept term, and its value indicates that the estimated amount of money that a casino makes on slot machines when they make `$0` from poker is `$13.8990`.
3. The `RAKE` coefficient indicates that for every 1\% increase in `RAKE`, `COIN-IN` increases by approximately 6.46\%. 
4. Day of the Week is a categorical variable with 7 categories, so even in a valid variable selection procedure, the one day of THU should not by itself be removed; either all days should be removed, or they should all be kept in.
5. `MAR 12` and `JUN 19` were included in the final model as dummy variables because these were outlier dates, and it is appropriate to use the data to determine whether to include them in the model for inference in this manner.
6. The choice of whether to include the AR(1) term in the model was based on whether an examination of correlograms indicated that serial correlation was removed with their inclusion, and this is a form of model selection that should not be done prior to inference

Store your selection(s) in the variable `final_model` below. If multiple answers are correct, store them as a vector of values in ascending order.

In [ ]:
final_model <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q4_5_1.R")

**Question 4.5.2.** Now, as much as we may not like how the final model was determined, let us try and run it on our data and compare the results we obtain to that of the paper. 

NOTE FOR BETA TESTERS (Valerie and Aditya): I am still trying to determine how to scaffold this. Some of that will depend on exactly how I present it in lecture (on Thursday of next week, and I haven't even started making the slides for that day yet). But for now I'm going to just put a full solution, and I'm open to any thoughts for how to do this.

In [ ]:
# Set up design matrix
X <- cbind(log(casino$R1_RAKE), 
                     ifelse(casino$dow=="Tuesday", 1, 0),
                     ifelse(casino$dow=="Wednesday", 1, 0),
                     ifelse(casino$dow=="Friday", 1, 0),
                     ifelse(casino$dow=="Saturday", 1, 0),
                     ifelse(casino$dow=="Sunday", 1, 0),
                     casino$STPATS,
          casino$MEMDAY,
          casino$INDDAY,
          ifelse(casino$date=="2009-03-12", 1, 0),
          ifelse(casino$date=="2009-06-19", 1, 0)
          )
colnames(X) <- c("RAKE", "TUE", "WED", "FRI", "SAT", "SUN", "STPATS", "MEMDAY", "INDDAY", "MAR12", "JUN19")

# Run model
ar1_arima <- Arima(
  y      = log(casino$R1_COIN),
  order  = c(1, 0, 0),
  xreg   = X,
  method = "ML"
)

# Show results
library(lmtest)
summary(ar1_arima)
coeftest(ar1_arima)  


Notice the similarity between this output and almost every term in the final model output. Good job Claude!

## Parting Thoughts

### History of Poker in Las Vegas
Here is a brief rundown of the history of poker in/relating to Las Vegas, somewhat from my own lense, but specifically as is relevant to this study:

 - 1970: First World Series of Poker (WSOP), held at Binion's Horseshoe Casino in downtown Las Vegas. According to [this source](https://www.pokernews.com/news/2017/07/history-world-series-of-poker-main-event-1970-1979-28376.htm?utm_source=chatgpt.com), there were only a few dozen poker *tables* in all of Las Vegas around that time.
 - 1998: Launch of first online poker site, Planet Poker. Although it stayed in business until 2017, it was never a major player in the online poker landscape, getting overshadowed by giants such as Paradise Poker (launched in 1999), PartyPoker (launched in 2001), PokerStars (launched in 2001), and Full Tilt Poker (launched in 2004).
 - 2003: First year that the WSOP was broadly televised, on ESPN. The WSOP Champion that year was Chris Moneymaker. The fact that he was an unassuming amateur who had won his seat into the WSOP from a \$40 buy-in online satellite event on PokerStars is credited with starting a massive poker boom ("If he can do it, so can I"), and it also didn't hurt for marketing purposes that he has a very fitting last name. For a handful of years subsequently, quite literally every casino property in Las Vegas opened a poker room, and also the number of entrants in the WSOP skyrocked:
    - 839 in 2003
    - 2,576 in 2004
    - 5,619 in 2005
    - 8,773 in 2006 (incidentally, the year that I won an online satellite into it and played)
 - 2006: Late in the year, the Unlawful Internet Gambling Enforcement Act (UIGEA) passes and goes into effect on October 13th of that year (a few months after that year's WSOP). Interestingly, UIGEA did not outlaw actual online gambling as a player, but it did outlaw operators from serving the USA. This discrepancy created a very fuzzy landscape, as players could still play with no legal consequences, but operators could disappear with no warning since they were now all operating illegally according to this act. The biggest online poker operator worldwide at that time, PartyPoker, did decide to withdraw from the USA as a result of UIGEA. Business volume at operators who decided to keep serving US-based customers dropped dramatically, as advertising for online poker stopped and customers had a lot of confusion and uncertainty about whether it was safe to play online poker. As online poker was a solid gateway to casino poker, this had the impact of curtailing some of the Las Vegas casino poker room business. One data point is the number of WSOP entrants in 2007: 6,358 (down from 8,773 in 2006, representing a substantial 27.5% decrease).
 - **2009: The year of the data in the paper of interest.**
     - Part of the motivation for this paper may have been the observation at this time that quite literally every property in Las Vegas has a poker room, and maybe this isn't optimal for every property from a business standpoint.
 - 2011: April 15th of that year is known in the poker community as "Black Friday." The DOJ launched a massive federal takedown of the 3 largest online poker operators at the time: PokerStars, Full Tilt Poker, and Absolute Poker/Ultimatebet. These sites had continued to operate in the USA despite UIGEA (mentioned above), and this takedown was the DOJ's response to that; each of their sites were abruptly taken down that day, and in their place, customers saw an ominous warning message from the DOJ. On a personal note, this resulted in approximately $4500 in my account on Full Tilt Poker being held in limbo until I (and all other customers) ultimately got it back two years later -- this was very stressful for me as that is no small amount of money! Some online poker operators did continue to service the USA despite this takedown of the 3 largest sites, but player traffic again dropped considerably. 
 - **2013: The year the paper of interest was published.**
 - 2015: The WSOP hits a local minimum in terms of number of entrants, of 6,420.
 - 2018: The first fully regulated and legalized online poker sites in the USA launch, in the states of Delaware, Nevada and New Jersey. Other states soon followed.
 - 2019: Number of WSOP entrants jumps from 7,221 in 2017 to 8,589 in 2019.
 - 2024: The WSOP hits its current maximum number of entrants, of 10,112. 

### Current State of Poker in Las Vegas
Claude repeatedly called this paper "prescient" in its conversations with me, in that 2009 was a peak in the number of Las Vegas properties that had poker rooms and it declined from there (114 in 2009, compared to 20 in 2026). However, there is a huge question of cause-and-effect, particularly as it relates to the legal status of online gambling and the impact of that on poker room business in Las Vegas. In other words, it is indisputable that the rise of poker in Las Vegas benefitted from the ability of US-based customers to freely play poker online, and legislation to restrict this had a 

Nevertheless, here is a rundown of the current state of Las Vegas poker rooms in 2026:

| Casino / Poker Room | Area | Approx. Poker Tables |
|---|---|---:|
| Venetian Las Vegas | Strip | 50 |
| Bellagio | Strip | 40 |
| The Palazzo at Venetian | Strip | 40 |
| The Orleans | Off-Strip | 34–35 |
| Venetian Resort | Strip | 30 |
| South Point | South Las Vegas | 30 |
| Caesars Palace | Strip | 28 |
| Wynn Las Vegas | Strip | 28 |
| Fontainebleau Las Vegas | Strip | 25 |
| Aria | Strip | 24 |
| MGM Grand | Strip | 22 |
| Westgate Las Vegas | Off-Strip | 20 |
| Santa Fe Station | Summerlin/NW | 14 |
| South Point (PokerAtlas listing) | South Las Vegas | 30 |
| Horseshoe Las Vegas (WSOP room) | Strip | ~18–20 |
| Mandalay Bay | Strip | ~10 |
| Sunset Station | Henderson | 10 |
| Boulder Station | East Las Vegas | 10 |
| Green Valley Ranch | Henderson | ~8 |
| Skyline Casino | Henderson | 2 |

From this, one conclusion is that it appears that poker in Las Vegas is now more centralized to 20 properties that mostly have very healthy poker rooms, as opposed to every property having a poker room of various sizes and sustainability.

## Congratulations, you are finished!!

To submit your assignment:

1. Select `Kernel -> Restart Kernel and Run All Cells...` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all tests passed.
3. Download your notebook using `File -> Download`, then upload your notebook to Gradescope.
4. Stick around while the Gradescope autograder grades your work. Make sure you see that all tests have passed on Gradescope.
5. Check that you have a confirmation email from Gradescope and save it as proof of your submission.